# Clase 7 · ¿Dónde cortamos la lista? Árboles, KNN, SVC y el umbral que paga

**Machine Learning** · Programa de Especialización Data Scientist · Módulo 2 · Sesión 7 de 15

Docente: **Josef Renato Rodriguez**

---

## Dónde quedamos

En la Clase 6 pegamos las cinco fuentes del banco, sacamos las fugas, cortamos por tiempo y entrenamos una
regresión logística: AUC **0.704** en el OOT de 2010. Pero con el umbral de 0.5 el modelo marcaba
«llamar» a solo **161 de 2 058** clientes.

Hoy, dos preguntas:

1. **¿Un modelo no lineal ordena mejor?** Árboles de clasificación, KNN y SVC, contra la logística.
2. **¿Dónde cortamos la lista** si cada llamada cuesta y cada venta deja margen?

## Qué vas a hacer hoy

| Bloque | Fase CRISP-DM | Contenido | Min |
|---|---|---|---|
| 1 | Modelar | El árbol a mano: Gini, entropía y profundidad | 18 |
| 2 | Modelar | KNN y SVC en bloque corto | 12 |
| 3 | Modelar | El desbalance: por qué el árbol no llama a nadie | 10 |
| 4 | Evaluar | La curva precision-recall y el PR-AUC | 8 |
| 5 | Evaluar · Desplegar | El umbral según el costo de la llamada | 20 |

Igual que siempre: completa las celdas **`TU CÓDIGO`** y ejecuta la **`VERIFICACIÓN`** de abajo.

---
## Celda 0 · Librerías, datos y la base de la Clase 6

La celda descarga `fuentes_banco.zip` del repositorio del curso (es el mismo de la Clase 6) y arma la base
con **la escalera de la Clase 6, ahora en una sola función**: mismas llaves, misma limpieza del maestro, mismo
rezago de la macro. Después corta por tiempo y entrena la regresión logística de la Clase 6, que es **el modelo
a batir**.

Si la descarga falla, sube el zip a mano con el botón de archivos de Colab y vuelve a ejecutar.

In [ ]:
import os, time, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, precision_recall_curve
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

# ── Datos: los mismos de la Clase 6 ──────────────────────────────────────────
URL = "https://raw.githubusercontent.com/josefrodrim/Super_ML_course/main/Clase_6/Data/fuentes_banco.zip"
if not os.path.exists("fuentes") and not os.path.exists("fuentes_banco.zip"):
    try:
        urllib.request.urlretrieve(URL, "fuentes_banco.zip")
        print("fuentes_banco.zip descargado del repositorio del curso")
    except Exception as e:
        print(f"No se pudo descargar ({e}). Súbelo a mano al panel de archivos de Colab.")
if not os.path.exists("fuentes") and os.path.exists("fuentes_banco.zip"):
    zipfile.ZipFile("fuentes_banco.zip").extractall(".")
if not os.path.exists("fuentes"):
    raise FileNotFoundError("Falta fuentes_banco.zip: súbelo al panel de archivos de Colab y vuelve a ejecutar esta celda.")

def construir_base(ruta="fuentes"):
    """La escalera de la Clase 6, completa, en una función."""
    gestiones = pd.read_csv(f"{ruta}/gestiones.csv", dtype={"cliente_id": str})
    historial = pd.read_csv(f"{ruta}/historial_campanas.csv")
    clientes = pd.read_csv(f"{ruta}/clientes.csv")
    macro = pd.read_csv(f"{ruta}/macro_mensual.csv")
    respuestas = pd.read_csv(f"{ruta}/respuestas.csv")
    base = gestiones.copy()
    base["cliente_id"] = base["cliente_id"].astype(int)                                            # peldaño 1
    base = base.merge(historial, on=["cliente_id", "periodo"], how="left", validate="one_to_one")
    unicos = clientes.sort_values("fecha_actualizacion").drop_duplicates("cliente_id", keep="last")  # peldaño 2
    base = base.merge(unicos, on="cliente_id", how="left", validate="many_to_one")
    base["fecha_llamada"] = pd.to_datetime(base["periodo"])                                         # peldaño 3
    macro["fecha_publicacion"] = pd.to_datetime(macro["periodo"]) + pd.offsets.MonthBegin(1)
    base = pd.merge_asof(base.sort_values("fecha_llamada"),
                         macro.rename(columns={"periodo": "periodo_macro"}).sort_values("fecha_publicacion"),
                         left_on="fecha_llamada", right_on="fecha_publicacion", direction="backward")
    base = base.merge(respuestas, on=["cliente_id", "periodo"], how="left", validate="one_to_one")   # peldaño 4
    base["contactado_antes"] = (base["dias_ultimo_contacto"] != 999).astype(int)
    return base.sort_values("gestion_id").reset_index(drop=True)

datos = construir_base()

# ── El corte temporal y las variables de la Clase 6 (sin fugas, sin macro) ──────
ent = datos[datos["periodo"] <= "2009-06"]
val = datos[(datos["periodo"] >= "2009-07") & (datos["periodo"] <= "2009-12")]
oot = datos[datos["periodo"] >= "2010-01"]
NUM = ["edad", "contactos_previos", "contactado_antes"]
CAT = ["ocupacion", "estado_civil", "educacion", "mora_previa", "hipoteca", "prestamo_consumo",
       "canal", "dia_semana", "resultado_previo"]
X = NUM + CAT

def preparar(escalar=True):
    """Imputa, escala las numéricas (si escalar=True) y codifica las categóricas."""
    num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()) if escalar else SimpleImputer(strategy="median")
    return ColumnTransformer([
        ("num", num, NUM),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="unknown"),
                              OneHotEncoder(handle_unknown="ignore")), CAT)])

def ks(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return (tpr - fpr).max()

def euros(x):
    return ("−€" if x < 0 else "€") + f"{abs(x):,.0f}".replace(",", " ")

# ── El modelo a batir: la regresión logística de la Clase 6 ─────────────────────
logit = Pipeline([("prep", preparar()), ("m", LogisticRegression(max_iter=2000))]).fit(ent[X], ent["contrato"])
p_val = logit.predict_proba(val[X])[:, 1]
p_oot = logit.predict_proba(oot[X])[:, 1]

for nombre, df in [("entrenamiento", ent), ("validación", val), ("OOT 2010", oot)]:
    print(f"{nombre:>13}: {len(df):>6} gestiones · tasa {df['contrato'].mean():.3f}")
print(f"\nRegresión logística de la Clase 6 · AUC validación {roc_auc_score(val['contrato'], p_val):.3f}"
      f" · AUC OOT {roc_auc_score(oot['contrato'], p_oot):.3f}")

---
# Bloque 1 · El árbol a mano: Gini, entropía y profundidad  ·  18 min

Un árbol de clasificación hace **preguntas de sí o no**, una tras otra: ¿ya lo contactamos antes? ¿lo llamamos
al celular? En cada paso elige **la pregunta que más ordena** a los clientes, es decir, la que deja grupos más
puros: casi todos contratan o casi nadie contrata.

La pureza se mide con la **impureza** de un grupo con tasa de contratación $p$:

$$\text{Gini}(p) = 1 - p^2 - (1-p)^2 \qquad\qquad \text{Entropía}(p) = -p\log_2 p - (1-p)\log_2(1-p)$$

Las dos valen **0** en un grupo puro y son máximas cuando $p = 0.5$. Una pregunta parte al grupo padre en dos
hijos, y su **ganancia** es cuánto baja la impureza:

$$\text{ganancia} = \text{impureza(padre)} - \left(\frac{n_{izq}}{n}\,\text{impureza(izq)} + \frac{n_{der}}{n}\,\text{impureza(der)}\right)$$

| Símbolo | Se lee | Qué es |
|---|---|---|
| $p$ | «pe» | la tasa de contratación del grupo |
| $n_{izq}/n$ | «ene izquierda sobre ene» | la proporción de clientes que van al hijo izquierdo |
| ganancia | | lo que se gana en pureza con la pregunta: el árbol elige la mayor |

### En papel primero

Diez llamadas. Contratan 4.

| Llamada | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| canal | cel | cel | cel | cel | cel | cel | fijo | fijo | fijo | fijo |
| contactado antes | sí | sí | no | no | no | sí | no | no | no | no |
| **contrató** | **1** | **1** | **1** | **1** | 0 | 0 | 0 | 0 | 0 | 0 |

Calcula a mano, antes de programar:

1. El Gini de la raíz: $p = 4/10$, así que $1 - 0.4^2 - 0.6^2 = $ ¿?
2. El corte por **canal**: celular tiene 4 de 6, fijo 0 de 4. ¿Cuál es el Gini de cada hijo y la ganancia?
3. El corte por **contactado antes**: sí tiene 2 de 3, no tiene 2 de 7. ¿Cuál gana?

### Ejercicio 1.1 — Las tres funciones, con el ejemplo de juguete

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
juguete = pd.DataFrame({"canal": ["celular"] * 6 + ["fijo"] * 4,
                        "contactado_antes": [1, 1, 0, 0, 0, 1, 0, 0, 0, 0],
                        "contrato":         [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]})

def gini(p):
    return None             # 1 - p**2 - (1 - p)**2

def entropia(p):
    if p == 0 or p == 1:
        return 0.0          # un grupo puro no tiene impureza (y log2(0) no existe)
    return None             # -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

def ganancia(y, izquierda, impureza):
    """y: 0/1 del grupo padre · izquierda: True para los que van al hijo izquierdo · impureza: gini o entropia."""
    y, izquierda = np.asarray(y), np.asarray(izquierda, dtype=bool)
    n = len(y)
    hijo_izq, hijo_der = y[izquierda], y[~izquierda]
    return None             # impureza(y.mean()) - (len(hijo_izq)/n * impureza(hijo_izq.mean()) + len(hijo_der)/n * impureza(hijo_der.mean()))

gini_raiz = None            # gini de la tasa de juguete["contrato"]
gan_gini_canal = None       # ganancia(juguete["contrato"], juguete["canal"] == "celular", gini)
gan_gini_contact = None     # ganancia(juguete["contrato"], juguete["contactado_antes"] == 1, gini)
gan_ent_canal = None        # lo mismo con entropia
gan_ent_contact = None

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("Gini de la raíz", gini_raiz, 0.4800),
     check("ganancia Gini · canal", gan_gini_canal, 0.2133),
     check("ganancia Gini · contactado antes", gan_gini_contact, 0.0610),
     check("ganancia entropía · canal", gan_ent_canal, 0.4200)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

La cuenta del canal, paso a paso: el celular tiene $p = 4/6$ y Gini $1 - (2/3)^2 - (1/3)^2 = 0.444$; el fijo es
puro (Gini 0). El promedio ponderado es $0.6 \cdot 0.444 + 0.4 \cdot 0 = 0.267$, y la ganancia, $0.48 - 0.267 = 0.213$.
El canal le gana a «contactado antes» (0.061) con los dos criterios: aquí el árbol empezaría
preguntando por el canal.

### Ejercicio 1.2 — La misma cuenta con los 36 939 clientes del entrenamiento

Ahora con los datos reales: ¿qué pregunta elige primero el árbol, «contactado antes» o «canal»?

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
y_ent = ent["contrato"]
izq_contact = ent["contactado_antes"] == 1
izq_canal = ent["canal"] == "cellular"

g_contact = None        # ganancia(y_ent, izq_contact, gini)
g_canal = None          # ganancia(y_ent, izq_canal, gini)
e_contact = None        # lo mismo con entropia
e_canal = None

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("ganancia Gini · contactado antes", g_contact, 0.002042, tol=1e-5),
     check("ganancia Gini · canal", g_canal, 0.001242, tol=1e-5),
     check("ganancia entropía · contactado antes", e_contact, 0.005807, tol=1e-5),
     check("ganancia entropía · canal", e_canal, 0.007119, tol=1e-5)]
print()
print("1.2 OK" if all(r) else "Revisa 1.2")

**Gini y entropía no eligen la misma pregunta.**

| Corte | Hijos | Gini | Entropía |
|---|---|---|---|
| contactado antes | 342 con tasa 40.4 % · 36 597 con 7.0 % | **0.00204** | 0.00581 |
| canal | 22 533 con tasa 9.3 % · 14 406 con 4.2 % | 0.00124 | **0.00712** |

- **Gini** prefiere aislar un grupo chico con tasa alta: los 342 ya contactados, que contratan el 40 %.
- **La entropía** es más sensible cuando las tasas son bajas: separar un 4.2 % de un 9.3 % le importa más, y elige el canal.

¿Cambia mucho? Vamos a verlo con `sklearn`.

### Demostración 1.3 — El árbol de `sklearn` confirma la cuenta

El árbol **no necesita escalar** las variables: solo compara cada una con un corte. Por eso usamos `preparar(escalar=False)`.

In [ ]:
# ── DEMOSTRACIÓN 1.3 ─────────────────────────────────────────────────────
arboles = {}
for criterio in ["gini", "entropy"]:
    arbol = Pipeline([("prep", preparar(escalar=False)),
                      ("m", DecisionTreeClassifier(max_depth=2, criterion=criterio, random_state=SEED))]).fit(ent[X], ent["contrato"])
    nombres = [n.split("__")[1] for n in arbol.named_steps["prep"].get_feature_names_out()]
    arboles[criterio] = arbol
    print(f"══ criterio = {criterio} ══")
    print(export_text(arbol.named_steps["m"], feature_names=nombres, show_weights=True))

print(f"¿A cuántos de los {len(ent)} clientes del entrenamiento manda llamar el árbol de Gini? "
      f"{arboles['gini'].predict(ent[X]).sum()}")

Tres cosas para leer en esa salida (`weights` es [no contrató, contrató] en cada hoja):

1. **La raíz es la que calculaste a mano:** `contactado_antes` con Gini, `canal_cellular` con entropía.
2. **Por caminos distintos llegan casi a las mismas hojas:** fijo sin contacto previo (tasa 4.1 %),
   celular sin contacto previo (8.8 %) y los ya contactados (alrededor del 40 %). El AUC en validación
   es 0.617 contra 0.617. En la práctica, el criterio rara vez es lo que decide.
3. **El árbol no manda llamar a nadie.** Todas las hojas tienen menos del 50 % de contratantes, así que en todas gana «no».
   Pero ordena: un cliente ya contactado tiene diez veces más chances que uno de fijo sin contacto. Volvemos a esto en el Bloque 3.

### Ejercicio 1.4 — ¿Qué tan profundo? La curva de sobreajuste

Un árbol sin límite sigue haciendo preguntas hasta que cada hoja es pura. Mide el AUC en entrenamiento y en
validación para cada profundidad, y **elige la profundidad con la validación**, nunca con el entrenamiento.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
profundidades = list(range(1, 16)) + [None]       # None = sin límite
filas = []
for prof in profundidades:
    arbol = Pipeline([("prep", preparar(escalar=False)),
                      ("m", DecisionTreeClassifier(max_depth=prof, random_state=SEED))]).fit(ent[X], ent["contrato"])
    filas.append({"profundidad": prof if prof is not None else "sin límite",
                  "hojas": arbol.named_steps["m"].get_n_leaves(),
                  "AUC entrenamiento": None,        # roc_auc_score(ent["contrato"], arbol.predict_proba(ent[X])[:, 1])
                  "AUC validación": None,           # lo mismo con val
                  "AUC OOT": roc_auc_score(oot["contrato"], arbol.predict_proba(oot[X])[:, 1])})
curva = pd.DataFrame(filas)

mejor_prof = None       # la profundidad con mayor AUC de validación, sin contar "sin límite"

In [ ]:
# ── VERIFICACIÓN 1.4 ─────────────────────────────────────────────────────
sin_limite = None if curva["AUC validación"].isna().any() else curva.iloc[-1]
r = [check("mejor profundidad", mejor_prof, 2, tol=0),
     check("AUC validación con esa profundidad", None if sin_limite is None else curva.loc[curva["profundidad"] == mejor_prof, "AUC validación"].iloc[0], 0.6166),
     check("AUC entrenamiento del árbol sin límite", None if sin_limite is None else sin_limite["AUC entrenamiento"], 0.9903),
     check("AUC validación del árbol sin límite", None if sin_limite is None else sin_limite["AUC validación"], 0.5233)]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.4")

In [ ]:
# ── DEMOSTRACIÓN 1.5 ─────────────────────────────────────────────────────
if not curva["AUC validación"].isna().any():
    c = curva[curva["profundidad"] != "sin límite"]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(c["profundidad"], c["AUC entrenamiento"], "o-", color=NAVY, label="entrenamiento")
    ax.plot(c["profundidad"], c["AUC validación"], "o-", color=BLUE, label="validación 2009")
    ax.plot(c["profundidad"], c["AUC OOT"], "o--", color=MAG, label="OOT 2010")
    ax.axvline(mejor_prof, color=GRAY, ls=":", label=f"elegida: {mejor_prof}")
    ax.set_xlabel("profundidad máxima del árbol"); ax.set_ylabel("AUC"); ax.set_xticks(range(1, 16))
    ax.set_title("Más preguntas: mejor en entrenamiento, peor en el futuro"); ax.legend()
    plt.tight_layout(); plt.show()

- El árbol **sin límite** tiene 5 963 hojas y 37 niveles. Memoriza el entrenamiento (AUC 0.990)
  y en validación da 0.523: casi azar.
- Desde la profundidad 3, cada nivel extra **sube** el entrenamiento y **baja** la validación y el OOT. Es el sobreajuste
  de la Clase 4, ahora en clasificación.
- Ganan **2 niveles y 4 hojas**: AUC 0.617 en validación y 0.700 en el OOT, casi lo mismo que
  la logística (0.619 y 0.704). Con estos datos la señal es simple, y cabe en cuatro hojas.

---
# Bloque 2 · KNN y SVC en bloque corto  ·  12 min

### KNN: dime con quién andas

Para puntuar a un cliente, KNN busca a los **k clientes más parecidos** del entrenamiento y devuelve la proporción
que contrató. No entrena nada: guarda la tabla y mide distancias al predecir.

| Decisión | Por qué importa |
|---|---|
| **escalar** | la distancia suma diferencias: sin escalar, 40 años de diferencia en la edad aplastan a todo lo demás |
| **k** | k chico memoriza (cada cliente se parece a su vecino exacto); k grande promedia |

### Ejercicio 2.1 — Elegir k en validación, y probar sin escalar

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
valores_k = [5, 15, 51, 101, 201, 401]
auc_knn = {}
for k in valores_k:
    knn = None                  # Pipeline([("prep", preparar(escalar=True)), ("m", KNeighborsClassifier(n_neighbors=k))]).fit(ent[X], ent["contrato"])
    auc_knn[k] = None           # AUC en validación: roc_auc_score(val["contrato"], knn.predict_proba(val[X])[:, 1])

mejor_k = None                  # el k con mayor AUC en validación: max(auc_knn, key=auc_knn.get)

knn_sin_escalar = None          # el mismo modelo con k = 201 pero con preparar(escalar=False)
auc_sin_escalar = None          # su AUC en validación

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("AUC validación con k = 5", auc_knn.get(5), 0.5695, tol=0.005),
     check("AUC validación con k = 201", auc_knn.get(201), 0.6163, tol=0.005),
     check("mejor k", mejor_k, 201, tol=0),
     check("AUC validación sin escalar (k = 201)", auc_sin_escalar, 0.5410, tol=0.005)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

- Con **k = 5** el AUC es 0.570: cinco vecinos son muy poca evidencia. Sube hasta **k = 201** (0.616) y se aplana.
- **Sin escalar** cae a 0.541: la edad domina la distancia y el resto de variables casi no cuenta.
- El KNN tiene un costo escondido: **para puntuar guarda y recorre las 36 939 filas**. Con millones de clientes
  y una campaña mensual, eso pesa.

### SVC: la frontera con el mayor margen

Ya conocen la idea por el SVR de la Clase 3: el SVC busca la frontera que separa a las dos clases **con el mayor margen**,
y el *kernel* le permite que esa frontera sea curva. Dos detalles prácticos:

| Detalle | Qué significa |
|---|---|
| no devuelve probabilidades | `decision_function` da la **distancia a la frontera**. Sirve para ordenar y para el AUC. `probability=True` las estima con validación cruzada interna y lo hace varias veces más lento |
| el costo crece rápido con n | el entrenamiento compara pares de clientes: duplicar los datos multiplica el tiempo **por 4 a 8** |

### Demostración 2.2 — El SVC con submuestras

In [ ]:
# ── DEMOSTRACIÓN 2.2 ─────────────────────────────────────────────────────
for n in [2500, 5000, 10000]:
    muestra, _ = train_test_split(ent, train_size=n, stratify=ent["contrato"], random_state=SEED)
    t0 = time.time()
    svc = Pipeline([("prep", preparar(escalar=True)), ("m", SVC(kernel="rbf", C=1))]).fit(muestra[X], muestra["contrato"])
    segundos = time.time() - t0
    auc = roc_auc_score(val["contrato"], svc.decision_function(val[X]))
    print(f"n = {n:>6}: {segundos:>5.1f} s · {svc.named_steps['m'].n_support_.sum():>5} vectores de soporte · AUC validación {auc:.3f}")

- El tiempo crece mucho más rápido que los datos: en esta corrida, 0.3 s, 1.0 s y 4.3 s.
  Con 20 000 clientes medimos unos 35 s, y con los 36 939 completos serían minutos.
- Y no gana: 0.587, 0.577 y 0.591, debajo de la logística (0.619). El kernel curvo no encuentra una
  estructura que la recta no haya visto. (El SVC con kernel lineal, por su parte, termina pareciéndose mucho a la logística.)

---
# Bloque 3 · El desbalance: por qué el árbol no llama a nadie  ·  10 min

En el entrenamiento solo contrata el **7.3 %**. Con tan pocos positivos, cualquier modelo aprende que
«no» casi siempre acierta. Por eso el árbol no mandó llamar a nadie, y la logística de la Clase 6, con el umbral
de 0.5, manda llamar a **21 de 36 939** en el entrenamiento.

La receta que se repite en internet es «balancea las clases». Veamos qué hace en realidad.

### Ejercicio 3.1 — El modelo que nunca llama, y la logística balanceada

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
exactitud_nunca = {}
for nombre, df in [("ent", ent), ("val", val), ("oot", oot)]:
    exactitud_nunca[nombre] = None      # la exactitud de predecir siempre 0: 1 - df["contrato"].mean()

logit_bal = None        # la logística de la Celda 0, pero con LogisticRegression(max_iter=2000, class_weight="balanced")
p_oot_bal = None        # sus probabilidades en el OOT
auc_oot_bal = None      # su AUC en el OOT
llama_bal = None        # a cuántos del OOT manda llamar con el umbral de 0.5: (p_oot_bal >= 0.5).sum()
rho = None              # correlación de orden entre p_oot y p_oot_bal: spearmanr(p_oot, p_oot_bal).correlation

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check("exactitud de no llamar nunca · entrenamiento", exactitud_nunca.get("ent"), 0.9270),
     check("AUC OOT de la logística balanceada", auc_oot_bal, 0.6922, tol=0.005),
     check("llamadas con 0.5, balanceada", llama_bal, 1683, tol=20),
     check("Spearman entre los dos scores", rho, 0.9792, tol=0.005)]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

**La exactitud no sirve con clases desbalanceadas.** No llamar a nadie acierta el 92.7 % en el entrenamiento,
y no vende nada.

**Balancear no mejora el orden: mueve las probabilidades.** `class_weight="balanced"` le da a cada contratante el peso de
unos 13 no contratantes. La probabilidad media en el OOT salta de 0.216 a 0.681 y, con el
0.5, pasa de llamar a 161 a llamar a 1 683. Pero el AUC queda igual (0.704 contra 0.692) y la correlación
de orden es 0.98: **ordena casi a los mismos clientes en el mismo orden**.

| Técnica | Qué hace | En nuestro caso |
|---|---|---|
| `class_weight="balanced"` | pesa más a los positivos | mismo orden, probabilidades infladas |
| submuestreo | descarta negativos hasta igualar | 5 390 filas: AUC 0.625 y 0.711, con el 15 % de los datos |
| SMOTE | inventa positivos interpolando vecinos | no lo usamos: agrega clientes que no existen y descalibra igual |

> Balancear es **mover el umbral por otra vía**, y de paso rompe la calibración. Es más limpio dejar las probabilidades
> como salen y **elegir el umbral a propósito**. Eso es el Bloque 5.

---
# Bloque 4 · La curva precision-recall y el PR-AUC  ·  8 min

La curva ROC compara a los contratantes contra **todos** los no contratantes. Cuando los positivos son pocos (fraude:
1 de cada mil), una ROC puede verse bien aunque los primeros de la lista estén llenos de falsos positivos.

La curva **precision-recall** mira solo lo que importa en una campaña: de los que llamo, cuántos contratan (precision),
según cuántos de los contratantes alcanzo (recall). Su área es el **PR-AUC** (en `sklearn`, `average_precision_score`).

| | ROC-AUC | PR-AUC |
|---|---|---|
| modelo al azar | 0.5, siempre | **la tasa de positivos** del conjunto |
| cambia si cambia la tasa | no | **sí** |

### Ejercicio 4.1 — PR-AUC contra su línea base, en los tres tramos

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
tabla_pr = {}
for nombre, df in [("entrenamiento", ent), ("validación", val), ("OOT", oot)]:
    p = logit.predict_proba(df[X])[:, 1]
    ap = None               # average_precision_score(df["contrato"], p)
    base = None             # la tasa de contratación del tramo
    tabla_pr[nombre] = {"ROC-AUC": roc_auc_score(df["contrato"], p), "PR-AUC": ap, "línea base": base,
                        "lift": None}       # PR-AUC / línea base

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("PR-AUC en validación", tabla_pr.get("validación", {}).get("PR-AUC"), 0.5396, tol=0.005),
     check("PR-AUC en OOT", tabla_pr.get("OOT", {}).get("PR-AUC"), 0.7026, tol=0.005),
     check("lift del PR-AUC en OOT", tabla_pr.get("OOT", {}).get("lift"), 1.3477, tol=0.01)]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

In [ ]:
# ── DEMOSTRACIÓN 4.2 ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
fpr, tpr, _ = roc_curve(oot["contrato"], p_oot)
axes[0].plot(fpr, tpr, color=BLUE, lw=2, label=f"modelo · ROC-AUC {roc_auc_score(oot['contrato'], p_oot):.3f}")
axes[0].plot([0, 1], [0, 1], color=GRAY, ls="--", label="azar · 0.5")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR (recall)"); axes[0].set_title("ROC · OOT 2010"); axes[0].legend()
prec, rec, _ = precision_recall_curve(oot["contrato"], p_oot)
axes[1].plot(rec, prec, color=MAG, lw=2, label=f"modelo · PR-AUC {average_precision_score(oot['contrato'], p_oot):.3f}")
axes[1].axhline(oot["contrato"].mean(), color=GRAY, ls="--", label=f"azar · tasa {oot['contrato'].mean():.3f}")
axes[1].set_xlabel("recall: contratantes alcanzados"); axes[1].set_ylabel("precision: de los llamados, contratan")
axes[1].set_title("Precision-recall · OOT 2010"); axes[1].set_ylim(0, 1.02); axes[1].legend()
plt.tight_layout(); plt.show()

El PR-AUC sube de 0.540 en validación a **0.703** en el OOT. Parece una mejora enorme, pero el lift es el mismo
(1.36 contra 1.35): 2010 simplemente tiene más contratantes (52.1 % contra 39.8 %).

> **Un PR-AUC solo se lee junto a su línea base.** No compares PR-AUC de períodos o carteras con tasas distintas.
> Úsalo cuando los positivos son raros y te importa la punta de la lista: fraude, lavado, default en carteras sanas.

---
# Bloque 5 · El umbral según el costo de la llamada  ·  20 min

### Los supuestos del caso

Los datos no traen costos: estos los fija el negocio, y hay que escribirlos.

| Supuesto | Valor | De dónde sale |
|---|---|---|
| costo de una gestión, $C$ | **€25** | en promedio 2.6 intentos, tiempo del agente, supervisión y sistema |
| margen de un depósito, $M$ | **€60** | el margen financiero de un año sobre el depósito promedio |

| | no llamar | llamar |
|---|---|---|
| **no contrata** | 0 | **−€25** (FP: llamada perdida) |
| **contrata** | 0 (FN: venta que no se hizo) | **+€35** (VP) |

$$\text{ganancia} = M \cdot VP - C \cdot (VP + FP)$$

### La regla teórica

Conviene llamar a un cliente si lo que espero ganar supera lo que cuesta: $p \cdot M > C$, es decir,

$$p > \frac{C}{M} = \frac{25}{60} = 0.417$$

Esa regla **solo vale si $p$ es una probabilidad bien calibrada**. Y en la Clase 6 vimos que no lo es: el modelo aprendió
en una época en que contrataba el 7 %, y en 2010 predice en promedio 0.216 cuando la tasa real es 0.521.

La alternativa empírica: **probar todos los umbrales en validación y quedarse con el que más gana**.

### Ejercicio 5.1 — La ganancia en validación con cuatro estrategias

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
COSTO, MARGEN = 25, 60

def ganancia_campana(y, score, umbral):
    llamar = score >= umbral
    vp = None               # contratantes entre los llamados: y[llamar].sum()
    return None             # MARGEN * vp - COSTO * llamar.sum()

y_val = val["contrato"].values
gan_todos_val = None        # llamar a todos: ganancia_campana(y_val, p_val, -1)
gan_05_val = None           # umbral 0.5
gan_cm_val = None           # umbral COSTO / MARGEN

umbrales = np.unique(p_val)
ganancias = None            # [ganancia_campana(y_val, p_val, u) for u in umbrales]
umbral_val = None           # el umbral con mayor ganancia: umbrales[np.argmax(ganancias)]
gan_mejor_val = None        # esa ganancia máxima

In [ ]:
# ── VERIFICACIÓN 5.1 ─────────────────────────────────────────────────────
r = [check("ganancia llamando a todos", gan_todos_val, -2455, tol=0),
     check("ganancia con umbral 0.5", gan_05_val, 1760, tol=100),
     check("ganancia con la regla C/M", gan_cm_val, 3040, tol=100),
     check("mejor umbral en validación", umbral_val, 0.2188, tol=0.01),
     check("ganancia máxima en validación", gan_mejor_val, 6070, tol=100)]
print()
print("5.1 OK" if all(r) else "Revisa 5.1")

- En el segundo semestre de 2009 contrata el 39.8 %, debajo del punto de equilibrio (41.7 %). **Llamar a todos pierde
  −€2 455.**
- El 0.5 llama a 100 y la regla teórica a 236: las dos llaman a muy pocos, porque las probabilidades del modelo son bajas
  (media 0.171 contra una tasa real de 0.398).
- El mejor umbral es **0.219**: llama a 494 clientes (el 23 %) y gana **€6 070**.

Ese umbral se eligió en 2009. La prueba de verdad es aplicarlo, sin tocarlo, a 2010.

### Ejercicio 5.2 — Las cuatro estrategias en el OOT de 2010

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
y_oot = oot["contrato"].values
gan_oot = {
    "umbral 0.5": None,                 # ganancia_campana(y_oot, p_oot, 0.5)
    "regla C/M": None,
    "umbral elegido en validación": None,
    "llamar a todos": None,
}

In [ ]:
# ── VERIFICACIÓN 5.2 ─────────────────────────────────────────────────────
r = [check("OOT · umbral 0.5", gan_oot["umbral 0.5"], 3415, tol=100),
     check("OOT · regla C/M", gan_oot["regla C/M"], 8575, tol=100),
     check("OOT · umbral elegido en validación", gan_oot["umbral elegido en validación"], 14455, tol=150),
     check("OOT · llamar a todos", gan_oot["llamar a todos"], 12930, tol=0)]
print()
print("5.2 OK" if all(r) else "Revisa 5.2")

In [ ]:
# ── DEMOSTRACIÓN 5.3 ─────────────────────────────────────────────────────
if umbral_val is not None:
    fig, ax = plt.subplots(figsize=(10, 4.3))
    for nombre, y, s, color in [("validación 2009", y_val, p_val, BLUE), ("OOT 2010", y_oot, p_oot, MAG)]:
        orden = np.argsort(-s)
        g = MARGEN * np.cumsum(y[orden]) - COSTO * np.arange(1, len(y) + 1)
        x = np.arange(1, len(y) + 1) / len(y)
        ax.plot(x, g, color=color, lw=2, label=nombre)
        i = (s >= umbral_val).sum() - 1
        ax.plot(x[i], g[i], "o", color=color, ms=9)
        print(f"{nombre}: con el umbral de validación llama al {x[i]:.0%} y gana {euros(g[i])} · el máximo posible era {euros(g.max())} llamando al {x[g.argmax()]:.0%}")
    ax.axhline(0, color=GRAY, lw=1)
    ax.set_xlabel("proporción de la lista que se llama (de mayor a menor score)"); ax.set_ylabel("ganancia (€)")
    ax.set_title("La curva de ganancia: los puntos son el umbral elegido en validación"); ax.legend()
    plt.tight_layout(); plt.show()

| Estrategia en 2010 | Llama a | Ganancia |
|---|---|---|
| umbral 0.5 (Clase 6) | 161 | €3 415 |
| regla teórica C/M | 413 | €8 575 |
| llamar a todos | 2 058 | €12 930 |
| **umbral elegido en validación** | **749** | **€14 455** |
| el máximo, mirando las respuestas (imposible de saber antes) | 1 512 | €17 280 |

- El umbral de validación gana **€14 455**: el 84 % del máximo que se podía sacar, y €11 040 más que el 0.5.
- **El umbral viaja con el score, no con el porcentaje.** El mismo 0.219 que en 2009 llamaba al 23 %, en 2010 llama al 36 %,
  porque en 2010 llegan clientes con mejor score. Si en cambio se fija la capacidad («llamo al 20 %»), el volumen no se adapta.
- En 2010 llamar a todos ya no pierde, porque la tasa subió al 52 %. Aun así el modelo gana más: deja de llamar a los que no compran.

### Demostración 5.4 — La tabla del día: cuatro modelos, un umbral cada uno

Para cada modelo: el umbral se elige en validación y se aplica sin tocar en 2010. El SVC no da probabilidades, pero
para elegir un umbral por ganancia **basta con un score que ordene**.

In [ ]:
# ── DEMOSTRACIÓN 5.4 ─────────────────────────────────────────────────────
muestra, _ = train_test_split(ent, train_size=10000, stratify=ent["contrato"], random_state=SEED)
modelos = {
    "Regresión logística": logit,
    f"Árbol (prof. {mejor_prof or 2})": Pipeline([("prep", preparar(escalar=False)),
                                         ("m", DecisionTreeClassifier(max_depth=mejor_prof or 2, random_state=SEED))]).fit(ent[X], ent["contrato"]),
    f"KNN (k = {mejor_k or 201})": Pipeline([("prep", preparar()), ("m", KNeighborsClassifier(n_neighbors=mejor_k or 201))]).fit(ent[X], ent["contrato"]),
    "SVC rbf (10 000)": Pipeline([("prep", preparar()), ("m", SVC(kernel="rbf", C=1))]).fit(muestra[X], muestra["contrato"]),
}
def gan(y, s, u):
    return MARGEN * y[s >= u].sum() - COSTO * (s >= u).sum()

filas = []
for nombre, m in modelos.items():
    puntuar = m.decision_function if nombre.startswith("SVC") else (lambda df, m=m: m.predict_proba(df)[:, 1])
    s_val, s_oot = puntuar(val[X]), puntuar(oot[X])
    us = np.unique(s_val)
    u = us[np.argmax([gan(y_val, s_val, x) for x in us])]
    filas.append({"modelo": nombre, "AUC val": roc_auc_score(y_val, s_val), "AUC OOT": roc_auc_score(y_oot, s_oot),
                  "PR-AUC OOT": average_precision_score(y_oot, s_oot), "KS OOT": ks(y_oot, s_oot),
                  "llama en OOT": int((s_oot >= u).sum()), "ganancia OOT (€)": int(gan(y_oot, s_oot, u))})
tabla_dia = pd.DataFrame(filas).set_index("modelo")
print(tabla_dia.round(3).to_string())

**El umbral vale más que el modelo.**

- Pasar del umbral 0.5 al umbral de validación, con la misma logística: **+€11 040**.
- Cambiar la logística por el árbol o el KNN, con su propio umbral: entre −€60 y −€130.
- El SVC queda último (€11 495), incluso debajo de llamar a todos.

Y el árbol de 2 niveles tiene una lectura que cualquier gerente entiende: su umbral (0.20) se traduce en
**«llamar a los que ya contactamos antes»**. Esa regla sola gana €14 395 en 2010, €60 menos que la logística.
Antes de pedir un modelo más complejo, hay que preguntarse cuánto le gana a la regla simple.

### Reto para quien termine antes

**Recalibrar en vez de buscar el umbral.** Ajusta en validación una regresión logística de una sola variable,
el logaritmo de las chances del score (`np.log(p_val / (1 - p_val))`), contra `y_val`. Aplica esa recalibración a `p_oot`,
usa la regla teórica $p > C/M$ y compara la ganancia en 2010 con la del umbral de validación.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Reto: tu código aquí

---
# Cierre

### Los números del día

| | AUC validación | AUC OOT | Ganancia OOT |
|---|---|---|---|
| Regresión logística, umbral 0.5 | 0.619 | 0.704 | €3 415 |
| **Regresión logística, umbral de validación** | 0.619 | 0.704 | **€14 455** |
| Árbol de profundidad 2 | 0.617 | 0.700 | €14 395 |
| KNN con k = 201 | 0.616 | 0.686 | €14 325 |
| SVC rbf, 10 000 clientes | 0.591 | 0.609 | €11 495 |
| Llamar a todos | | | €12 930 |

### Checklist de salida

- [ ] Calculo a mano el Gini y la entropía de un grupo, y la ganancia de un corte.
- [ ] Elijo la profundidad de un árbol (y el k de un KNN) con la validación, nunca con el entrenamiento.
- [ ] Escalo antes de KNN y SVC; el árbol no lo necesita.
- [ ] Sé que el SVC da un score, no una probabilidad, y que su costo crece rápido con los datos.
- [ ] Con clases desbalanceadas, no miro la exactitud, y sé que balancear mueve las probabilidades, no el orden.
- [ ] Leo el PR-AUC junto a su línea base.
- [ ] Escribo los costos del negocio y elijo el umbral que más gana en validación.

### Próxima clase · Viernes 16/10

**Clase 8 · Random Forest y XGBoost para clasificación, y repaso.** Juntamos muchos árboles: ¿le ganan por fin a la
logística en el OOT de 2010? Cerramos el Módulo 2 con la comparativa completa y un repaso para el **parcial del lunes 19/10**.